In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import joblib


In [2]:
df = pd.read_csv('medical.csv')

In [3]:
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


Custom Interaction Feature Transformer

In [4]:
def add_interactions(X):
    X = X.copy()
    X["bmi_smoker"] = X["bmi"] * (X["smoker"] == "yes").astype(int)
    X["age_smoker"] = X["age"] * (X["smoker"] == "yes").astype(int)
    return X


Define Columns

In [5]:
num_features = ['age', 'bmi', 'children']
cat_features = ['sex', 'smoker', 'region']


Preprocessing Block

In [6]:
preprocessor = Pipeline(steps=[
    ("interactions", FunctionTransformer(add_interactions)),
    ("transform", ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_features),
            ("cat", OneHotEncoder(drop="first"), cat_features)
        ],
        remainder="passthrough"
    ))
])


Full Pipeline

In [7]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])


Train & Evaluate

In [8]:
from sklearn.model_selection import train_test_split

X = df.drop('charges', axis=1)
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_pipeline.fit(X_train, y_train)

joblib.dump(model_pipeline, "medical_insurance_pipeline.pkl")


['medical_insurance_pipeline.pkl']

Evaluation